# MRA Databricks Cluster Sanity Check

Reusable validation notebook for a newly provisioned Databricks cluster.

Checks basic Python/Spark execution, package availability (including XGBoost and MLflow), Unity Catalog access, `mer_wealth_dev` access, and a small scikit-learn training test.

In [ ]:
results = {}
print('===== BASIC CLUSTER TEST =====')
try:
    print('Cluster is working')
    results['Basic Python'] = ('PASS', 'Python execution successful')
except Exception as e:
    results['Basic Python'] = ('FAIL', str(e))

In [ ]:
print('===== CLUSTER / RUNTIME CHECK =====')
import sys
print('Python version:')
print(sys.version)
print('\nSpark version:')
print(spark.version)
print('\nSpark master:')
print(spark.sparkContext.master)

In [ ]:
print('===== SPARK TEST =====')
try:
    spark.range(10).show()
    results['Spark'] = ('PASS', f'Spark {spark.version} executed successfully')
except Exception as e:
    results['Spark'] = ('FAIL', str(e))

In [ ]:
print('===== PACKAGE AVAILABILITY CHECK =====')
packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scikit-learn': 'sklearn',
    'mlflow': 'mlflow',
    'xgboost': 'xgboost'
}
for display_name, import_name in packages.items():
    try:
        module = __import__(import_name)
        version = getattr(module, '__version__', 'Version not available')
        print(f'{display_name:15} : INSTALLED - {version}')
        results[display_name] = ('PASS', f'Installed - {version}')
    except Exception as e:
        print(f'{display_name:15} : NOT AVAILABLE - {e}')
        results[display_name] = ('FAIL', str(e))

In [ ]:
print('===== UNITY CATALOG TEST =====')
try:
    catalogs_df = spark.sql('SHOW CATALOGS')
    catalogs_df.show(truncate=False)
    results['Unity Catalog'] = ('PASS', 'SHOW CATALOGS executed successfully')
except Exception as e:
    results['Unity Catalog'] = ('FAIL', str(e))

In [ ]:
print('===== MRA CATALOG CHECK =====')
expected_catalog = 'mer_wealth_dev'
try:
    available_catalogs = [row.catalog for row in spark.sql('SHOW CATALOGS').collect()]
    if expected_catalog in available_catalogs:
        print(f"SUCCESS: Catalog '{expected_catalog}' is accessible.")
        spark.sql(f'SHOW SCHEMAS IN {expected_catalog}').show(truncate=False)
        results['mer_wealth_dev access'] = ('PASS', 'Catalog and schemas accessible')
    else:
        print(f"WARNING: Catalog '{expected_catalog}' is not accessible.")
        results['mer_wealth_dev access'] = ('FAIL', 'Catalog not visible')
except Exception as e:
    results['mer_wealth_dev access'] = ('FAIL', str(e))

In [ ]:
print('===== SCIKIT-LEARN MODEL TEST =====')
try:
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score
    data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, test_size=0.2, random_state=42
    )
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print('Model trained successfully')
    print('Test accuracy:', accuracy)
    results['Scikit-learn training'] = ('PASS', f'Accuracy={accuracy:.4f}')
except Exception as e:
    results['Scikit-learn training'] = ('FAIL', str(e))

In [ ]:
print('===== FINAL VALIDATION SUMMARY =====')
summary_rows = [(name, status, details) for name, (status, details) in results.items()]
summary_df = spark.createDataFrame(summary_rows, ['Test', 'Status', 'Details'])
display(summary_df)
failed = [name for name, (status, _) in results.items() if status == 'FAIL']
print('\n===================================')
if not failed:
    print('ALL VALIDATION CHECKS PASSED')
else:
    print('VALIDATION COMPLETED WITH FAILURES')
    print('Failed checks:', ', '.join(failed))
print('===================================')